In [2]:
import math
import colorsys

# ==================================================
# HUE BAR CALIBRATION
# ==================================================

# Hue anchors from left -> right, every 8 taps.
# Anchor 25 is 1 tap away from the far-right red edge.
HUE_ANCHORS = [
    "#FE0000", "#FF0086", "#FF00B8", "#FF00DA", "#FF00FA",
    "#E700FF", "#C700FF", "#9A00FF", "#5300FF", "#006DFF",
    "#00A9FF", "#00D0FF", "#00F1FF", "#00FFF1", "#00FFD3",
    "#00FFAB", "#00FF73", "#48FF00", "#96FF00", "#C2FF00",
    "#E6FF00", "#FFFD00", "#FFDF00", "#FFBB00", "#FF8C00",
    "#FF3100"
]

HUE_TAPS_PER_ANCHOR = 8

# Since the last anchor is 1 tap away from the right edge.
# Anchor 25 = tap 200.
# Far right red edge = tap 201.
MAX_HUE_TAPS = 201

# Search small hue offsets around each anchor.
# You can widen this if needed, but -4..4 should be enough.
HUE_OFFSET_SEARCH = range(-4, 5)


# ==================================================
# SATURATION / BRIGHTNESS RECTANGLE CALIBRATION
# This was measured for the red hue.
# ==================================================

# Horizontal rectangle positions, left -> right
X_POSITIONS = [0, 37, 69, 109, 149, 181, 213]  # Segment 0..6

# Vertical rectangle positions, top -> bottom
Y_POSITIONS = [0, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 113]

# Segment grid, top -> bottom for each horizontal segment.
# Segment 0 = grayscale side
# Segment 6 = full red side
SEGMENT_GRID = [
    # Segment 0, grayscale
    [
        "#FDFDFD", "#F6F6F6", "#EEEEEA", "#E5E5E5", "#D6D6D4",
        "#D2D2D2", "#C7C7C5", "#BCBCBA", "#B0B0AE", "#A3A3A1",
        "#939391", "#828282", "#6D6C6F", "#515151", "#000000"
    ],

    # Segment 1
    [
        "#FFEFEF", "#F6E8E7", "#EEE0DF", "#E5D7D6", "#DCCECE",
        "#D2C4C3", "#C7B9B8", "#BCAEAE", "#AFA4A2", "#A49598",
        "#928887", "#827676", "#6D6362", "#514849", "#000000"
    ],

    # Segment 2
    [
        "#FFDBDB", "#F6D4D2", "#EECCCB", "#E6C4C3", "#DCBCBD",
        "#D2B3B1", "#C8A6A5", "#BB9D9D", "#B09292", "#A48887",
        "#927A78", "#826D6C", "#6D5B5B", "#504241", "#000000"
    ],

    # Segment 3
    [
        "#FFBEBC", "#F7B7B7", "#EEB0B1", "#E6AAA9", "#DBA3A2",
        "#D29C9A", "#C79493", "#BC8C8C", "#B08282", "#A47879",
        "#946C6C", "#825E5E", "#6D4F4D", "#503A3C", "#000000"
    ],

    # Segment 4
    [
        "#FF9A98", "#F69491", "#EE8F8D", "#E58A87", "#DB8381",
        "#D27D7A", "#C77673", "#BC6E6C", "#B06665", "#A35D5D",
        "#935351", "#834948", "#6D3C38", "#512929", "#000000"
    ],

    # Segment 5
    [
        "#FF766E", "#F77269", "#EE6E65", "#E56961", "#DB645C",
        "#D25F58", "#C75A53", "#BC554C", "#AF5147", "#A34940",
        "#924237", "#813B31", "#6D2E25", "#51221C", "#000000"
    ],

    # Segment 6, full red side
    [
        "#FF3000", "#F62D01", "#EE2E01", "#E52A00", "#DB2A00",
        "#D22602", "#C82600", "#BB2500", "#B02100", "#A31C00",
        "#931600", "#821500", "#6D0D00", "#4D0E00", "#000000"
    ]
]


# ==================================================
# BASIC COLOR HELPERS
# ==================================================

def normalize_hex(hex_color):
    hex_color = hex_color.strip().upper()

    if not hex_color.startswith("#"):
        hex_color = "#" + hex_color

    if len(hex_color) != 7:
        raise ValueError(f"Invalid hex: {hex_color}")

    return hex_color


def hex_to_rgb(hex_color):
    hex_color = normalize_hex(hex_color)

    return (
        int(hex_color[1:3], 16),
        int(hex_color[3:5], 16),
        int(hex_color[5:7], 16),
    )


def rgb_to_hex(rgb):
    r = max(0, min(255, round(rgb[0])))
    g = max(0, min(255, round(rgb[1])))
    b = max(0, min(255, round(rgb[2])))

    return f"#{r:02X}{g:02X}{b:02X}"


def rgb_distance(a, b):
    return math.sqrt(
        (a[0] - b[0]) ** 2 +
        (a[1] - b[1]) ** 2 +
        (a[2] - b[2]) ** 2
    )


def lerp(a, b, t):
    return a + (b - a) * t


def lerp_rgb(c1, c2, t):
    return tuple(lerp(c1[i], c2[i], t) for i in range(3))


def find_interval(value, anchors):
    if value <= anchors[0]:
        return 0, 0.0

    if value >= anchors[-1]:
        return len(anchors) - 2, 1.0

    for i in range(len(anchors) - 1):
        a = anchors[i]
        b = anchors[i + 1]

        if a <= value <= b:
            if b == a:
                return i, 0.0
            return i, (value - a) / (b - a)

    return len(anchors) - 2, 1.0


# ==================================================
# HSV / HUE HELPERS
# ==================================================

def rgb_to_hsv_tuple(rgb):
    r, g, b = rgb
    return colorsys.rgb_to_hsv(r / 255.0, g / 255.0, b / 255.0)


def hsv_to_rgb_tuple(h, s, v):
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return (r * 255.0, g * 255.0, b * 255.0)


def hue_distance(h1, h2):
    """
    Circular hue distance, where hue is 0.0..1.0.
    """
    d = abs(h1 - h2)
    return min(d, 1.0 - d)


def hue_from_taps(hue_taps):
    """
    Estimate hue from the measured hue anchors.

    This interpolates between neighboring hue anchors in HSV hue space.
    """
    hue_taps = max(0, min(MAX_HUE_TAPS, hue_taps))

    anchor_hues = [rgb_to_hsv_tuple(hex_to_rgb(h))[0] for h in HUE_ANCHORS]

    # Special case: after final anchor to far-right red edge.
    # Far-right edge is treated as red hue 0.0.
    if hue_taps >= (len(HUE_ANCHORS) - 1) * HUE_TAPS_PER_ANCHOR:
        last_tap = (len(HUE_ANCHORS) - 1) * HUE_TAPS_PER_ANCHOR
        t = (hue_taps - last_tap) / (MAX_HUE_TAPS - last_tap)

        h1 = anchor_hues[-1]
        h2 = 0.0

        # Interpolate circularly from orange-red to red.
        if h2 < h1:
            h2 += 1.0

        h = (h1 + (h2 - h1) * t) % 1.0
        return h

    left_index = hue_taps // HUE_TAPS_PER_ANCHOR
    right_index = left_index + 1

    t = (hue_taps - (left_index * HUE_TAPS_PER_ANCHOR)) / HUE_TAPS_PER_ANCHOR

    h1 = anchor_hues[left_index]
    h2 = anchor_hues[right_index]

    # circular interpolation
    if abs(h2 - h1) > 0.5:
        if h1 > h2:
            h2 += 1.0
        else:
            h1 += 1.0

    h = (h1 + (h2 - h1) * t) % 1.0
    return h


def recolor_red_rgb_to_hue(red_grid_rgb, target_hue):
    """
    Keep the calibrated red grid's saturation/value-ish behavior,
    but swap its hue to target_hue.

    Grayscale-ish colors have low saturation, so hue barely matters there.
    """
    h, s, v = rgb_to_hsv_tuple(red_grid_rgb)
    return hsv_to_rgb_tuple(target_hue, s, v)


# ==================================================
# RECTANGLE COLOR INTERPOLATION
# ==================================================

def red_grid_color_at(x, y):
    """
    Bilinear interpolation in the measured red rectangle.
    """
    xi, tx = find_interval(x, X_POSITIONS)
    yi, ty = find_interval(y, Y_POSITIONS)

    c00 = hex_to_rgb(SEGMENT_GRID[xi][yi])
    c10 = hex_to_rgb(SEGMENT_GRID[xi + 1][yi])
    c01 = hex_to_rgb(SEGMENT_GRID[xi][yi + 1])
    c11 = hex_to_rgb(SEGMENT_GRID[xi + 1][yi + 1])

    top = lerp_rgb(c00, c10, tx)
    bottom = lerp_rgb(c01, c11, tx)

    return lerp_rgb(top, bottom, ty)


def color_at(x, y, hue_taps):
    """
    Estimated color at a rectangle position x/y after choosing hue_taps.
    """
    target_hue = hue_from_taps(hue_taps)
    red_rgb = red_grid_color_at(x, y)
    return recolor_red_rgb_to_hue(red_rgb, target_hue)


# ==================================================
# CORNER LOGIC FOR RECTANGLE
# ==================================================

def closest_corner(x, y):
    max_x = X_POSITIONS[-1]
    max_y = Y_POSITIONS[-1]

    corners = {
        "top_left": x + y,
        "top_right": (max_x - x) + y,
        "bottom_left": x + (max_y - y),
        "bottom_right": (max_x - x) + (max_y - y),
    }

    return min(corners, key=corners.get)


def taps_from_corner(x, y, corner):
    max_x = X_POSITIONS[-1]
    max_y = Y_POSITIONS[-1]

    if corner == "top_left":
        return f"{x} taps right, {y} taps down"

    if corner == "top_right":
        return f"{max_x - x} taps left, {y} taps down"

    if corner == "bottom_left":
        return f"{x} taps right, {max_y - y} taps up"

    if corner == "bottom_right":
        return f"{max_x - x} taps left, {max_y - y} taps up"

    raise ValueError("Invalid corner")


# ==================================================
# HUE ANCHOR OUTPUT LOGIC
# ==================================================

def hue_instruction_from_taps(hue_taps):
    """
    Convert absolute hue tap position into:
    Anchor N + K taps right/left

    It uses whichever nearby anchor is closer.
    """
    hue_taps = max(0, min(MAX_HUE_TAPS, hue_taps))

    anchor_positions = [
        i * HUE_TAPS_PER_ANCHOR
        for i in range(len(HUE_ANCHORS))
    ]

    # Add far-right red edge as a possible reference.
    # This is not technically one of your sampled anchors,
    # but it is a physical edge.
    references = [(i, pos) for i, pos in enumerate(anchor_positions)]
    references.append(("right edge", MAX_HUE_TAPS))

    best_ref, best_pos = min(
        references,
        key=lambda item: abs(hue_taps - item[1])
    )

    diff = hue_taps - best_pos

    if diff == 0:
        if best_ref == "right edge":
            return "Hue: right edge"
        return f"Hue: Anchor {best_ref}"

    direction = "right" if diff > 0 else "left"
    amount = abs(diff)

    if best_ref == "right edge":
        return f"Hue: right edge + {amount} taps {direction}"

    return f"Hue: Anchor {best_ref} + {amount} taps {direction}"


def likely_hue_taps_for_target(target_rgb):
    """
    Estimate likely hue tap positions for target color.
    Returns a small list of candidate hue taps near matching hue anchors.

    For low-saturation colors like gray/black/white, hue does not matter much,
    so include a few common candidates.
    """
    target_h, target_s, target_v = rgb_to_hsv_tuple(target_rgb)

    # Grays/blacks/whites: hue is meaningless.
    # Use red anchor by default, plus a few extras just in case.
    if target_s < 0.03:
        return [0, MAX_HUE_TAPS, 100]

    anchor_hues = [rgb_to_hsv_tuple(hex_to_rgb(h))[0] for h in HUE_ANCHORS]

    candidates = []

    for i, h in enumerate(anchor_hues):
        base_taps = i * HUE_TAPS_PER_ANCHOR
        candidates.append((hue_distance(target_h, h), base_taps))

    # Include far-right red edge too.
    candidates.append((hue_distance(target_h, 0.0), MAX_HUE_TAPS))

    candidates.sort(key=lambda x: x[0])

    # Take the 4 closest anchor/edge areas, then search offsets around them.
    tap_candidates = set()

    for _, base_taps in candidates[:4]:
        for offset in HUE_OFFSET_SEARCH:
            tap = base_taps + offset
            if 0 <= tap <= MAX_HUE_TAPS:
                tap_candidates.add(tap)

    return sorted(tap_candidates)


# ==================================================
# MAIN SOLVER
# ==================================================

def find_best_position(target_hex):
    target_rgb = hex_to_rgb(target_hex)
    hue_candidates = likely_hue_taps_for_target(target_rgb)

    best = {
        "x": 0,
        "y": 0,
        "hue_taps": 0,
        "rgb": None,
        "dist": float("inf"),
    }

    for hue_taps in hue_candidates:
        for x in range(X_POSITIONS[0], X_POSITIONS[-1] + 1):
            for y in range(Y_POSITIONS[0], Y_POSITIONS[-1] + 1):
                rgb = color_at(x, y, hue_taps)
                d = rgb_distance(target_rgb, rgb)

                if d < best["dist"]:
                    best = {
                        "x": x,
                        "y": y,
                        "hue_taps": hue_taps,
                        "rgb": rgb,
                        "dist": d,
                    }

    return best


# ==================================================
# OUTPUT
# ==================================================

def describe_hex(hex_color):
    hex_color = normalize_hex(hex_color)
    best = find_best_position(hex_color)

    x = best["x"]
    y = best["y"]
    hue_taps = best["hue_taps"]

    corner = closest_corner(x, y)
    rect_instruction = taps_from_corner(x, y, corner)
    hue_instruction = hue_instruction_from_taps(hue_taps)

    predicted = rgb_to_hex(best["rgb"])

    print(
        f"{hex_color} -> "
        f"{hue_instruction}; "
        f"Rectangle: start from {corner.replace('_', ' ')}: {rect_instruction}. "
        f"Predicted: {predicted}. "
        f"Error: {best['dist']:.2f}"
    )


def describe_hex_list(hex_list):
    for hex_color in hex_list:
        describe_hex(hex_color)


# ==================================================
# EXAMPLE
# ==================================================

if __name__ == "__main__":
    test_colors = [
        "#7A3C38",
        "#939393",
        "#3B62A3",
        "#B148FF",
        "#48FF00",
        "#FFB6C1",
        "#2E8B57",
        "#FFD700",
        "#111111"
    ]

    describe_hex_list(test_colors)

#7A3C38 -> Hue: Anchor 0; Rectangle: start from bottom right: 52 taps left, 22 taps up. Predicted: #7A3A3A. Error: 2.84
#939393 -> Hue: Anchor 0; Rectangle: start from bottom left: 0 taps right, 34 taps up. Predicted: #959393. Error: 2.00
#3B62A3 -> Hue: Anchor 9 + 1 taps left; Rectangle: start from bottom right: 30 taps left, 42 taps up. Predicted: #3D5FA5. Error: 3.53
#B148FF -> Hue: Anchor 7 + 1 taps right; Rectangle: start from top right: 21 taps left, 0 taps down. Predicted: #B048FF. Error: 0.79
#48FF00 -> Hue: Anchor 17; Rectangle: start from top right: 0 taps left, 0 taps down. Predicted: #48FF00. Error: 0.00
#FFB6C1 -> Hue: Anchor 0 + 2 taps right; Rectangle: start from top right: 98 taps left, 0 taps down. Predicted: #FFB7C0. Error: 1.07
#2E8B57 -> Hue: Anchor 16; Rectangle: start from bottom right: 28 taps left, 29 taps up. Predicted: #2E8A57. Error: 1.46
#FFD700 -> Hue: Anchor 22 + 2 taps right; Rectangle: start from top right: 0 taps left, 0 taps down. Predicted: #FFD600. E

In [6]:
mazda_colors = [
"#8D9394",
"#616363",
"#6D747B",
"#525453",
"#444444",
"#323333",
"#2E3944",
"#232323",
"#131313",
"#152534",
"#132D43",
"#010101",
]
describe_hex_list(mazda_colors)

#8D9394 -> Hue: Anchor 11 + 1 taps left; Rectangle: start from bottom left: 23 taps right, 34 taps up. Predicted: #8D9395. Error: 0.55
#616363 -> Hue: Anchor 12 + 4 taps right; Rectangle: start from bottom left: 3 taps right, 14 taps up. Predicted: #616363. Error: 0.26
#6D747B -> Hue: Anchor 9 + 4 taps right; Rectangle: start from bottom left: 43 taps right, 22 taps up. Predicted: #6D747A. Error: 0.89
#525453 -> Hue: Anchor 12 + 4 taps right; Rectangle: start from bottom left: 10 taps right, 10 taps up. Predicted: #525454. Error: 1.58
#444444 -> Hue: Anchor 0; Rectangle: start from bottom left: 18 taps right, 8 taps up. Predicted: #484444. Error: 4.00
#323333 -> Hue: Anchor 0; Rectangle: start from bottom left: 18 taps right, 6 taps up. Predicted: #363333. Error: 4.00
#2E3944 -> Hue: Anchor 9; Rectangle: start from bottom right: 90 taps left, 8 taps up. Predicted: #2E3947. Error: 3.43
#232323 -> Hue: Anchor 0; Rectangle: start from bottom left: 9 taps right, 4 taps up. Predicted: #2423